In [ ]:
3.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

# --- 1. CONFIGURACIÓN: 3 QUBITS, 4 CLASES ---
N_QUBITS = 3
DIM = 2**N_QUBITS
np.random.seed(42)

print(f"Iniciando simulación para N={N_QUBITS} qubits, DIM={DIM}).")
print("Clases: 0=W, 1=GHZ, 2=Dicke(k=2), 3=GHZ-Impostor (fase)")
print("Ruido: Amplitud T1 (para verificar inicio)")

# --- 2. NUEVA FÍSICA: Canales Cuánticos (Operadores de Kraus) ---

def get_amplitude_damping_kraus(gamma, n_qubits):
    K0_1q = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1_1q = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = [];
        for op in kraus_ops:
            for k in kraus_1q: new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops

def apply_channel(rho, kraus_ops):
    rho_final = np.zeros_like(rho, dtype=complex)
    for K in kraus_ops:
        rho_final += K @ rho @ K.conj().T
    # Simple check for trace, T1 preserves trace mostly
    trace = np.trace(rho_final).real
    if not np.isclose(trace, 1.0) and np.abs(trace)>1e-9:
        rho_final = rho_final / trace
    return rho_final

print("Funciones de Canal Cuántico (Kraus) definidas.")

# --- 3. Generación de Estados Puros (N=3) ---

def ghz3_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = 1/np.sqrt(2)
    return np.outer(psi, psi.conj())

def w3_state():
    psi = np.zeros(DIM);
    for i in range(N_QUBITS): psi[1 << i] = 1/np.sqrt(N_QUBITS)
    return np.outer(psi, psi.conj())

def dicke_state_rho(n_qubits, k):
    psi = np.zeros(2**n_qubits)
    indices = []
    for i in range(2**n_qubits):
        if bin(i).count('1') == k: indices.append(i)
    if not indices: return np.zeros((DIM, DIM))
    norm = 1.0 / np.sqrt(len(indices))
    for i in indices: psi[i] = norm
    if k==2: print(f"Estado Dicke |D_2^3> creado con 3 términos.")
    return np.outer(psi, psi.conj())

def ghz_impostor_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = -1/np.sqrt(2)
    return np.outer(psi, psi.conj())

# --- 4. Extracción de Features (N=3) ---

X_1 = np.array([[0, 1], [1, 0]]); X_N = X_1
for _ in range(N_QUBITS - 1): X_N = np.kron(X_N, X_1)
print("Operador global X(N=3) pre-calculado.")

def shannon_entropy(prob):
    prob_clean = prob[prob > 1e-15]; return -np.sum(prob_clean * np.log2(prob_clean))

def get_rho_q(rho, q, n_qubits=N_QUBITS):
    rho_q = np.zeros((2, 2), dtype=complex)
    mask = 1 << (n_qubits - 1 - q)
    for i in range(DIM):
        for j in range(DIM):
            if (i & ~mask) == (j & ~mask):
                bi = (i >> (n_qubits - 1 - q)) & 1; bj = (j >> (n_qubits - 1 - q)) & 1
                rho_q[bi, bj] += rho[i, j]
    return rho_q

def partial_entropy(rho_q):
    eigs = np.linalg.eigvalsh(rho_q); eigs = np.clip(eigs, 1e-15, 1)
    return -np.sum(eigs * np.log2(eigs))

def extract_features(rho):
    probabilities_Z = np.diag(rho).real
    H_Z = shannon_entropy(probabilities_Z)
    H_partial = []
    for q in range(N_QUBITS):
        rho_q_i = get_rho_q(rho, q)
        H_partial.append(partial_entropy(rho_q_i))
    H_q_avg = np.mean(H_partial)
    E_X = np.trace(rho @ X_N).real
    return np.array([H_Z, E_X, H_q_avg])

# --- 5. La Simulación (Solo necesitamos t=0) ---

# Estados puros iniciales (t=0)
states_pure = {
    "W (k=1)": w3_state(),
    "GHZ": ghz3_state(),
    "Dicke D(2,3)": dicke_state_rho(N_QUBITS, 2),
    "Impostor": ghz_impostor_state()
}

# Calculamos las features SOLO para los estados puros
initial_features = {}
print("\nCalculando features iniciales (t=0)...")
for name, rho_pure in states_pure.items():
    features = extract_features(rho_pure)
    initial_features[name] = features
    print(f"{name:<15}: H_Z={features[0]:.4f}, E_X={features[1]:.4f}, H_q={features[2]:.4f}")

colors = {"W (k=1)": "blue", "GHZ": "red", "Dicke D(2,3)": "green", "Impostor": "black"}

# --- 6. ¡NUEVO GRÁFICO 2D! ---

print("\nGenerando Gráfico 2D de Falla Estática (N=3)...")

plt.figure(figsize=(10, 7))

# Ploteamos SOLO los puntos iniciales
for name, features in initial_features.items():
    plt.scatter(features[0], features[2], # H_Z vs H_q_avg
                color=colors[name],
                marker='x', # Usamos 'x' para marcar el inicio
                s=200,      # Tamaño grande
                label=f'{name} (Inicio Puro t=0)')

plt.title('GRÁFICO PARA SECCIÓN 3.1: Falla Estática (N=3)', fontsize=15)
plt.xlabel('Entropía en Base Z (H_Z)')
plt.ylabel('Entropía Parcial Promedio (H_q Avg)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Añadimos texto si queremos resaltar el solapamiento
plt.text(1.6, 0.92, '¡W y Dicke solapados!', fontsize=12, color='purple', ha='center')

plt.show()

In [ ]:
3.2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

# --- 1. CONFIGURACIÓN: 3 QUBITS, 4 CLASES ---
N_QUBITS = 3
DIM = 2**N_QUBITS
np.random.seed(42)

print(f"Iniciando simulación para N={N_QUBITS} qubits, DIM={DIM}).")
print("Clases: 0=W, 1=GHZ, 2=Dicke(k=2), 3=GHZ-Impostor (fase)")
print("Ruido: Amplitud T1 (para verificar inicio)")

# --- 2. NUEVA FÍSICA: Canales Cuánticos (Operadores de Kraus) ---

def get_amplitude_damping_kraus(gamma, n_qubits):
    K0_1q = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1_1q = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = [];
        for op in kraus_ops:
            for k in kraus_1q: new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops

def apply_channel(rho, kraus_ops):
    rho_final = np.zeros_like(rho, dtype=complex)
    for K in kraus_ops:
        rho_final += K @ rho @ K.conj().T
    # Simple check for trace, T1 preserves trace mostly
    trace = np.trace(rho_final).real
    if not np.isclose(trace, 1.0) and np.abs(trace)>1e-9:
        rho_final = rho_final / trace
    return rho_final

print("Funciones de Canal Cuántico (Kraus) definidas.")

# --- 3. Generación de Estados Puros (N=3) ---

def ghz3_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = 1/np.sqrt(2)
    return np.outer(psi, psi.conj())

def w3_state():
    psi = np.zeros(DIM);
    for i in range(N_QUBITS): psi[1 << i] = 1/np.sqrt(N_QUBITS)
    return np.outer(psi, psi.conj())

def dicke_state_rho(n_qubits, k):
    psi = np.zeros(2**n_qubits)
    indices = []
    for i in range(2**n_qubits):
        if bin(i).count('1') == k: indices.append(i)
    if not indices: return np.zeros((DIM, DIM))
    norm = 1.0 / np.sqrt(len(indices))
    for i in indices: psi[i] = norm
    if k==2: print(f"Estado Dicke |D_2^3> creado con 3 términos.")
    return np.outer(psi, psi.conj())

def ghz_impostor_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = -1/np.sqrt(2)
    return np.outer(psi, psi.conj())

# --- 4. Extracción de Features (N=3) ---

X_1 = np.array([[0, 1], [1, 0]]); X_N = X_1
for _ in range(N_QUBITS - 1): X_N = np.kron(X_N, X_1)
print("Operador global X(N=3) pre-calculado.")

def shannon_entropy(prob):
    prob_clean = prob[prob > 1e-15]; return -np.sum(prob_clean * np.log2(prob_clean))

def get_rho_q(rho, q, n_qubits=N_QUBITS):
    rho_q = np.zeros((2, 2), dtype=complex)
    mask = 1 << (n_qubits - 1 - q)
    for i in range(DIM):
        for j in range(DIM):
            if (i & ~mask) == (j & ~mask):
                bi = (i >> (n_qubits - 1 - q)) & 1; bj = (j >> (n_qubits - 1 - q)) & 1
                rho_q[bi, bj] += rho[i, j]
    return rho_q

def partial_entropy(rho_q):
    eigs = np.linalg.eigvalsh(rho_q); eigs = np.clip(eigs, 1e-15, 1)
    return -np.sum(eigs * np.log2(eigs))

def extract_features(rho):
    probabilities_Z = np.diag(rho).real
    H_Z = shannon_entropy(probabilities_Z)
    H_partial = []
    for q in range(N_QUBITS):
        rho_q_i = get_rho_q(rho, q)
        H_partial.append(partial_entropy(rho_q_i))
    H_q_avg = np.mean(H_partial)
    E_X = np.trace(rho @ X_N).real
    return np.array([H_Z, E_X, H_q_avg])

# --- 5. La Simulación (Solo necesitamos t=0) ---

# Estados puros iniciales (t=0)
states_pure = {
    "W (k=1)": w3_state(),
    "GHZ": ghz3_state(),
    "Dicke D(2,3)": dicke_state_rho(N_QUBITS, 2),
    "Impostor": ghz_impostor_state()
}

# Calculamos las features SOLO para los estados puros
initial_features = {}
print("\nCalculando features iniciales (t=0)...")
for name, rho_pure in states_pure.items():
    features = extract_features(rho_pure)
    initial_features[name] = features
    print(f"{name:<15}: H_Z={features[0]:.4f}, E_X={features[1]:.4f}, H_q={features[2]:.4f}")

colors = {"W (k=1)": "blue", "GHZ": "red", "Dicke D(2,3)": "green", "Impostor": "black"}

# --- 6. ¡NUEVO GRÁFICO 2D! ---

print("\nGenerando Gráfico 2D de Falla Estática (N=3)...")

plt.figure(figsize=(10, 7))

# Ploteamos SOLO los puntos iniciales
for name, features in initial_features.items():
    plt.scatter(features[0], features[2], # H_Z vs H_q_avg
                color=colors[name],
                marker='x', # Usamos 'x' para marcar el inicio
                s=200,      # Tamaño grande
                label=f'{name} (Inicio Puro t=0)')

plt.title('GRÁFICO PARA SECCIÓN 3.1: Falla Estática (N=3)', fontsize=15)
plt.xlabel('Entropía en Base Z (H_Z)')
plt.ylabel('Entropía Parcial Promedio (H_q Avg)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)

# Añadimos texto si queremos resaltar el solapamiento
plt.text(1.6, 0.92, '¡W y Dicke solapados!', fontsize=12, color='purple', ha='center')

plt.show()

In [ ]:
3.3

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# --- 1. CONFIGURACIÓN: 3 QUBITS, 2 TIPOS DE RUIDO ---
N_QUBITS = 3
DIM = 2**N_QUBITS
np.random.seed(42)

print(f"Iniciando simulación 'FORENSE' (N={N_QUBITS} qubits, DIM={DIM}).")
print("Comparando 2 'Causas de Muerte': Amplitud (T1) vs. Fase (T2)")

# --- 2. CANALES CUÁNTICOS (KRAUS) ---

def get_amplitude_damping_kraus(gamma, n_qubits):
    """ Ruido T1: Pérdida de Energía (|1> -> |0>) """
    K0_1q = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1_1q = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = []
        for op in kraus_ops:
            for k in kraus_1q:
                new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops

def get_phase_damping_kraus(gamma, n_qubits):
    """ ¡NUEVO RUIDO T2: Pérdida de Fase! """
    # Esto ataca las coherencias (off-diagonals)
    # K0 = [[1, 0], [0, sqrt(1-g)]]
    # K1 = [[0, 0], [0, sqrt(g)]] * Z = [[0, 0], [0, -sqrt(g)]]  <- ¡ERROR!
    # K1 = [[0, 0], [0, sqrt(g)]] <-- Esto es Phase *Flip*
    # El T2 puro (de-faseo) es:
    K0_1q = np.array([[1, 0], [0, np.sqrt(1-gamma)]], dtype=complex)
    K1_1q = np.array([[1, 0], [0, -np.sqrt(1-gamma)]], dtype=complex) # ¡ERROR!
    
    # --- Corrección de la física ---
    # Los operadores de De-faseo (Phase Damping T2) correctos son:
    # K0 = [[1, 0], [0, sqrt(1-gamma)]]
    # K1 = [[0, 0], [0, sqrt(gamma)]] <-- ¡No!
    # K0 = I
    # K1 = sqrt(gamma) * Z
    # ¡No!
    # Los correctos son:
    K0_1q = np.eye(2, dtype=complex)
    K0_1q[1,1] = np.sqrt(1 - gamma)
    K1_1q = np.zeros((2,2), dtype=complex)
    K1_1q[1,1] = np.sqrt(gamma)
    # ¡NO! Esos son para phase *flip*.
    
    # OK, la forma más simple de Phase Damping T2 (mata off-diagonals)
    # rho_final[0,1] = rho_inicial[0,1] * (1-gamma)
    # Es K0 = [[1, 0], [0, 1-gamma]] ? No.
    # Es K0 = I, K1 = Z ? No.
    
    # ¡Aquí están! K0 = [[1,0],[0,1]], K1 = [[sqrt(g),0],[0,0]], K2 = [[0,sqrt(g)],[0,0]] ??
    # ¡Es increíblemente difícil encontrar los correctos!
    
    # Vamos a usar el modelo T2 estándar:
    # rho -> (1-p) * rho + p * Z*rho*Z
    # No, ese es Phase Flip.
    
    # OK, el canal de De-faseo (T2) que solo mata coherencia es:
    # K0 = [[1, 0], [0, sqrt(1-gamma)]]
    # K1 = [[0, 0], [0, sqrt(gamma)]]  <-- ¡Esto es Amplitude Damping sobre el |1>!
    
    # ¡¡YA SÉ!! Es K0 = I, y se aplica K_t = diag(1, e^(-gamma*t))
    # No, eso no es Kraus.
    
    # ¡AQUÍ ESTÁ!
    K0_1q = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1_1q = np.array([[0, 0], [0, 0]], dtype=complex) 
    # ¡No!
    
    # Último intento: K0 = [[1, 0], [0, 1]], K1 = [[0, 0], [0, sqrt(gamma)]]
    # ¡No!
    
    # OK, vamos a usar el canal de PHASE *FLIP*. Es un tipo de T2.
    # K0 = sqrt(1-g) * I
    # K1 = sqrt(g) * Z
    K0_1q = np.sqrt(1 - gamma) * np.eye(2, dtype=complex)
    K1_1q = np.sqrt(gamma) * np.array([[1, 0], [0, -1]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = []
        for op in kraus_ops:
            for k in kraus_1q:
                new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops


def apply_channel(rho, kraus_ops):
    rho_final = np.zeros_like(rho, dtype=complex)
    for K in kraus_ops:
        rho_final += K @ rho @ K.conj().T
    return rho_final

print("Canales de Ruido (Amplitud T1 y Fase T2) definidos.")

# --- 3. Generación de Estados Puros (N=3) ---
# (NECESITAMOS LOS VECTORES 'psi' para la Fidelidad)

def ghz3_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = 1/np.sqrt(2)
    return psi, np.outer(psi, psi.conj())

def w3_state():
    psi = np.zeros(DIM);
    for i in range(N_QUBITS): psi[1 << i] = 1/np.sqrt(N_QUBITS)
    return psi, np.outer(psi, psi.conj())

def dicke_state(n_qubits, k):
    psi = np.zeros(2**n_qubits)
    indices = []
    for i in range(2**n_qubits):
        if bin(i).count('1') == k: indices.append(i)
    if not indices: return psi, np.zeros((DIM, DIM))
    norm = 1.0 / np.sqrt(len(indices))
    for i in indices: psi[i] = norm
    if k==2: print(f"Estado Dicke |D_2^3> creado con 3 términos.")
    return psi, np.outer(psi, psi.conj())

def ghz_impostor_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = -1/np.sqrt(2)
    return psi, np.outer(psi, psi.conj())

# --- 4. Extracción de Features (¡con Fidelidad!) ---

X_1 = np.array([[0, 1], [1, 0]]); X_N = X_1
for _ in range(N_QUBITS - 1): X_N = np.kron(X_N, X_1)
print("Operador global X(N=3) pre-calculado.")

def shannon_entropy(prob):
    prob_clean = prob[prob > 1e-15]; return -np.sum(prob_clean * np.log2(prob_clean))

def get_rho_q(rho, q, n_qubits=N_QUBITS):
    rho_q = np.zeros((2, 2), dtype=complex)
    mask = 1 << (n_qubits - 1 - q)
    for i in range(DIM):
        for j in range(DIM):
            if (i & ~mask) == (j & ~mask):
                bi = (i >> (n_qubits - 1 - q)) & 1; bj = (j >> (n_qubits - 1 - q)) & 1
                rho_q[bi, bj] += rho[i, j]
    return rho_q

def partial_entropy(rho_q):
    eigs = np.linalg.eigvalsh(rho_q); eigs = np.clip(eigs, 1e-15, 1) 
    return -np.sum(eigs * np.log2(eigs))

def calculate_fidelity(psi_pure, rho_decayed):
    """ Mide qué tan 'parecido' es rho_decayed al psi_pure original """
    # F(psi, rho) = <psi|rho|psi>
    fid = psi_pure.conj().T @ rho_decayed @ psi_pure
    return fid.real # Debe ser real

def extract_features_and_stats(rho_decayed, psi_pure):
    # Métricas del Mapa
    probabilities_Z = np.diag(rho_decayed).real
    H_Z = shannon_entropy(probabilities_Z)
    H_partial_list = []
    for q in range(N_QUBITS):
        rho_q_i = get_rho_q(rho_decayed, q)
        H_partial_list.append(partial_entropy(rho_q_i))
    H_q_avg = np.mean(H_partial_list)
    E_X = np.trace(rho_decayed @ X_N).real
    
    # ¡NUEVA ESTADÍSTICA!
    Fidelity = calculate_fidelity(psi_pure, rho_decayed)
    
    return np.array([H_Z, E_X, H_q_avg, Fidelity])

# --- 5. La Simulación (¡Doble Simulación!) ---

N_STEPS = 50
gamma_steps = np.linspace(0, 1.0, N_STEPS)

# Estados puros iniciales (t=0)
states_pure_dict = {
    "W (k=1)": w3_state(),
    "GHZ": ghz3_state(),
    "Dicke D(2,3)": dicke_state(N_QUBITS, 2),
    "Impostor": ghz_impostor_state()
}

# ¡Guardaremos dos sets de películas!
trajectories_T1_Amplitude = {name: [] for name in states_pure_dict}
trajectories_T2_Phase = {name: [] for name in states_pure_dict}
colors = {"W (k=1)": "blue", "GHZ": "red", "Dicke D(2,3)": "green", "Impostor": "black"}

print(f"Iniciando DOBLE simulación de {N_STEPS} pasos de tiempo...")
start_time = time.time()

for gamma in gamma_steps:
    # 1. Obtenemos las "reglas de muerte" para este paso de tiempo
    kraus_T1 = get_amplitude_damping_kraus(gamma, N_QUBITS)
    kraus_T2 = get_phase_damping_kraus(gamma, N_QUBITS)
    
    for name, (psi_pure, rho_pure) in states_pure_dict.items():
        # --- Simulación 1: Muerte por Amplitud (T1) ---
        rho_T1 = apply_channel(rho_pure, kraus_T1)
        features_T1 = extract_features_and_stats(rho_T1, psi_pure)
        trajectories_T1_Amplitude[name].append(features_T1)
        
        # --- Simulación 2: Muerte por Fase (T2) ---
        rho_T2 = apply_channel(rho_pure, kraus_T2)
        features_T2 = extract_features_and_stats(rho_T2, psi_pure)
        trajectories_T2_Phase[name].append(features_T2)

end_time = time.time()
print(f"Doble simulación completada en {end_time - start_time:.2f} segundos.")

# Convertimos a arrays de numpy
for name in states_pure_dict:
    trajectories_T1_Amplitude[name] = np.array(trajectories_T1_Amplitude[name])
    trajectories_T2_Phase[name] = np.array(trajectories_T2_Phase[name])

# --- 6. REGISTRO DE DATOS y GRÁFICOS ESTADÍSTICOS ---

print("Generando panel de 'Estadísticas Forenses'...")

# Creamos una figura con 3 filas y 2 columnas
fig, axes = plt.subplots(3, 2, figsize=(14, 18))
fig.suptitle('Análisis Forense: Comparación de "Causas de Muerte" (Ruido T1 vs T2)', fontsize=20, y=1.02)

# --- Columna 1: Ruido de AMPLITUD (T1) ---
ax_T1_fid = axes[0, 0]
ax_T1_hq = axes[1, 0]
ax_T1_ex = axes[2, 0]

ax_T1_fid.set_title('Muerte por Amplitud (T1 - Energía)', fontsize=15)
ax_T1_fid.set_ylabel('Fidelidad (Supervivencia)', fontsize=12)
ax_T1_hq.set_ylabel('H_q (Entrelazamiento)', fontsize=12)
ax_T1_ex.set_ylabel('E_X (Correlación Fase)', fontsize=12)
ax_T1_ex.set_xlabel('Tiempo de Muerte (gamma)', fontsize=12)

# --- Columna 2: Ruido de FASE (T2) ---
ax_T2_fid = axes[0, 1]
ax_T2_hq = axes[1, 1]
ax_T2_ex = axes[2, 1]

ax_T2_fid.set_title('Muerte por Fase (T2 - Coherencia)', fontsize=15)
ax_T2_fid.set_ylabel('Fidelidad (Supervivencia)', fontsize=12)
ax_T2_hq.set_ylabel('H_q (Entrelazamiento)', fontsize=12)
ax_T2_ex.set_ylabel('E_X (Correlación Fase)', fontsize=12)
ax_T2_ex.set_xlabel('Tiempo de Muerte (gamma)', fontsize=12)

# Columnas de features
# [0] = H_Z, [1] = E_X, [2] = H_q_avg, [3] = Fidelity
col_fid = 3
col_hq = 2
col_ex = 1

# Llenamos los gráficos
for name, color in colors.items():
    # Plots de T1 (Amplitud)
    path_T1 = trajectories_T1_Amplitude[name]
    ax_T1_fid.plot(gamma_steps, path_T1[:, col_fid], color=color, label=name)
    ax_T1_hq.plot(gamma_steps, path_T1[:, col_hq], color=color, label=name)
    ax_T1_ex.plot(gamma_steps, path_T1[:, col_ex], color=color, label=name)

    # Plots de T2 (Fase)
    path_T2 = trajectories_T2_Phase[name]
    ax_T2_fid.plot(gamma_steps, path_T2[:, col_fid], color=color, label=name)
    ax_T2_hq.plot(gamma_steps, path_T2[:, col_hq], color=color, label=name)
    ax_T2_ex.plot(gamma_steps, path_T2[:, col_ex], color=color, label=name)

# Añadimos leyendas y grillas
for ax_row in axes:
    for ax in ax_row:
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)
        
plt.tight_layout(rect=[0, 0.03, 1, 0.98])
plt.show()

# --- 7. REGISTRO DE DATOS (LOG) ---
print("\n" + "="*75)
print("--- REGISTRO FORENSE (LOG) ---")
print("="*75)

for name in states_pure_dict:
    print(f"\n--- Estado: {name} ---")
    path_T1 = trajectories_T1_Amplitude[name]
    path_T2 = trajectories_T2_Phase[name]
    
    print("                |      RUIDO T1 (AMPLITUD)      |      RUIDO T2 (FASE)")
    print("Paso (gamma)    | Fidelidad | H_q    | E_X       | Fidelidad | H_q    | E_X")
    print("----------------+-----------+--------+-----------+-----------+--------+-----------")
    
    # Imprimimos t=0, t=0.2, t=0.5, t=1.0
    indices = [0, N_STEPS // 5, N_STEPS // 2, N_STEPS - 1]
    gammas = [gamma_steps[i] for i in indices]
    
    for i, g in zip(indices, gammas):
        f_t1 = path_T1[i]
        f_t2 = path_T2[i]
        print(f"t={g:<14.2f} | {f_t1[col_fid]:<9.3f} | {f_t1[col_hq]:<6.3f} | {f_t1[col_ex]:<9.3f} | {f_t2[col_fid]:<9.3f} | {f_t2[col_hq]:<6.3f} | {f_t2[col_ex]:<9.3f}")

print("="*75 + "\n")

In [ ]:
3.4

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import time

# --- 1. CONFIGURACIÓN: 5 QUBITS, 4 CLASES, RUIDO MIXTO 80/20 ---
N_QUBITS = 5
DIM = 2**N_QUBITS
np.random.seed(42)

print(f"Iniciando simulación de ROBUSTEZ (N={N_QUBITS} qubits, DIM={DIM}).")
print("Clases: 0=W, 1=GHZ, 2=Dicke(k=2), 3=GHZ-Impostor (fase)")
print("Ruido: MIXTO (80% Amplitud T1 + 20% Fase T2)") # <-- CAMBIO AQUÍ

# --- 2. CANALES CUÁNTICOS (KRAUS) ---
# (Las funciones get_amplitude_damping_kraus y get_phase_flip_kraus son las mismas)
def get_amplitude_damping_kraus(gamma_t1, n_qubits):
    K0_1q = np.array([[1, 0], [0, np.sqrt(1 - gamma_t1)]], dtype=complex)
    K1_1q = np.array([[0, np.sqrt(gamma_t1)], [0, 0]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = [];
        for op in kraus_ops:
            for k in kraus_1q: new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops

def get_phase_flip_kraus(gamma_t2, n_qubits):
    K0_1q = np.sqrt(1 - gamma_t2) * np.eye(2, dtype=complex)
    K1_1q = np.sqrt(gamma_t2) * np.array([[1, 0], [0, -1]], dtype=complex)
    kraus_1q = [K0_1q, K1_1q]
    kraus_ops = [np.array([1], dtype=complex)]
    for _ in range(n_qubits):
        new_ops = [];
        for op in kraus_ops:
            for k in kraus_1q: new_ops.append(np.kron(op, k))
        kraus_ops = new_ops
    return kraus_ops

def apply_channel(rho, kraus_ops):
    rho_final = np.zeros_like(rho, dtype=complex)
    for K in kraus_ops:
        rho_final += K @ rho @ K.conj().T
    trace = np.trace(rho_final)
    if not np.isclose(trace, 1.0):
       if np.abs(trace) > 1e-9: rho_final = rho_final / trace
       else: rho_final = np.zeros_like(rho, dtype=complex); rho_final[0,0] = 1.0
    return rho_final

print("Funciones de Canal Cuántico (Kraus) definidas para T1 y T2.")

# --- 3. Generación de Estados Puros (N=5) ---
# (Las funciones son las mismas)
def ghz5_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = 1/np.sqrt(2)
    return np.outer(psi, psi.conj())

def w5_state():
    psi = np.zeros(DIM);
    for i in range(N_QUBITS): psi[1 << i] = 1/np.sqrt(N_QUBITS)
    return np.outer(psi, psi.conj())

def dicke_state_rho(n_qubits, k):
    psi = np.zeros(2**n_qubits)
    indices = []
    for i in range(2**n_qubits):
        if bin(i).count('1') == k: indices.append(i)
    if not indices: return np.zeros((DIM, DIM))
    norm = 1.0 / np.sqrt(len(indices))
    for i in indices: psi[i] = norm
    if k==2: print(f"Estado Dicke |D_2^5> creado con {len(indices)} términos (10).")
    return np.outer(psi, psi.conj())

def ghz_impostor_state():
    psi = np.zeros(DIM); psi[0] = 1/np.sqrt(2); psi[-1] = -1/np.sqrt(2)
    return np.outer(psi, psi.conj())

# --- 4. Extracción de Features (N=5) ---
# (Las funciones son las mismas)
X_1 = np.array([[0, 1], [1, 0]]); X_N = X_1
for _ in range(N_QUBITS - 1): X_N = np.kron(X_N, X_1)
print("Operador global X(N=5) pre-calculado.")

def shannon_entropy(prob):
    prob_clean = prob[prob > 1e-15]; return -np.sum(prob_clean * np.log2(prob_clean))

def get_rho_q(rho, q, n_qubits=N_QUBITS):
    rho_q = np.zeros((2, 2), dtype=complex)
    mask = 1 << (n_qubits - 1 - q)
    for i in range(DIM):
        for j in range(DIM):
            if (i & ~mask) == (j & ~mask):
                bi = (i >> (n_qubits - 1 - q)) & 1; bj = (j >> (n_qubits - 1 - q)) & 1
                rho_q[bi, bj] += rho[i, j]
    return rho_q

def partial_entropy(rho_q):
    eigs = np.linalg.eigvalsh(rho_q); eigs = np.clip(eigs, 1e-15, 1)
    return -np.sum(eigs * np.log2(eigs))

def extract_features(rho):
    probabilities_Z = np.diag(rho).real
    H_Z = shannon_entropy(probabilities_Z)
    H_partial = []
    for q in range(N_QUBITS):
        rho_q_i = get_rho_q(rho, q)
        H_partial.append(partial_entropy(rho_q_i))
    H_q_avg = np.mean(H_partial)
    E_X = np.trace(rho @ X_N).real
    return np.array([H_Z, E_X, H_q_avg])

# --- 5. La Simulación (¡CON RUIDO MIXTO 80/20!) ---

N_STEPS = 50
gamma_total_max = 1.0
gamma_steps = np.linspace(0, gamma_total_max, N_STEPS)

# --- ¡CAMBIO EN LA MEZCLA DE RUIDO! ---
t1_fraction = 0.8 # 80% Amplitud T1
t2_fraction = 0.2 # 20% Fase T2
# --- ---

states_pure = {
    "W (k=1)": w5_state(),
    "GHZ": ghz5_state(),
    "Dicke D(2,5)": dicke_state_rho(N_QUBITS, 2),
    "Impostor": ghz_impostor_state()
}

trajectories_Mixed_8020 = {name: [] for name in states_pure} # Nuevo diccionario
colors = {"W (k=1)": "blue", "GHZ": "red", "Dicke D(2,5)": "green", "Impostor": "black"}

print(f"Iniciando simulación de {N_STEPS} pasos con Ruido Mixto 80/20...")
start_time = time.time()

for gamma_total in gamma_steps:
    gamma_t1 = gamma_total * t1_fraction
    gamma_t2 = gamma_total * t2_fraction
    
    kraus_T1 = get_amplitude_damping_kraus(gamma_t1, N_QUBITS)
    kraus_T2 = get_phase_flip_kraus(gamma_t2, N_QUBITS)
    
    for name, rho_pure in states_pure.items():
        rho_temp = apply_channel(rho_pure, kraus_T1)
        rho_mixed_decay = apply_channel(rho_temp, kraus_T2)
        
        features = extract_features(rho_mixed_decay)
        trajectories_Mixed_8020[name].append(features)

end_time = time.time()
print(f"Simulación completada en {end_time - start_time:.2f} segundos.")

# Convertimos a arrays
for name in trajectories_Mixed_8020:
    trajectories_Mixed_8020[name] = np.array(trajectories_Mixed_8020[name])

# Calculamos el punto final teórico (estado |0...0>)
rho_muerto = np.zeros((DIM, DIM)); rho_muerto[0, 0] = 1
features_muerto = extract_features(rho_muerto)

# --- 6. REGISTRO DE DATOS (LOG) ---

print("\n" + "="*65)
print("--- LOG DE TRAYECTORIAS (RUIDO MIXTO 80/20, N=5) ---") # <-- Título actualizado
print("="*65)
print(f"{'Estado':<15} {'Paso':<10} {'H_Z':<10} {'E_X':<10} {'H_q Avg':<10}")
print("-" * 65)

for name, path in trajectories_Mixed_8020.items():
    start_features = path[0]
    mid_features = path[N_STEPS // 2] # Gamma = 0.5
    end_features = path[-1] # Gamma = 1.0
    
    print(f"{name:<15} {'INICIO':<10} {start_features[0]:<10.3f} {start_features[1]:<10.3f} {start_features[2]:<10.3f}")
    print(f"{name:<15} {'MITAD':<10} {mid_features[0]:<10.3f} {mid_features[1]:<10.3f} {mid_features[2]:<10.3f}")
    print(f"{name:<15} {'FINAL':<10} {end_features[0]:<10.3f} {end_features[1]:<10.3f} {end_features[2]:<10.3f}")
    print("-" * 65)

print(f"{'MUERTE (T1)':<15} {'N/A':<10} {features_muerto[0]:<10.3f} {features_muerto[1]:<10.3f} {features_muerto[2]:<10.3f}") # Recordatorio del final T1 puro
print("="*65 + "\n")


# --- 7. Visualización (Mapa 3D con Ruido Mixto 80/20) ---

print("Generando el Mapa 3D con Ruido Mixto 80/20...")

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

for name, path in trajectories_Mixed_8020.items():
    ax.plot(path[:, 0], path[:, 1], path[:, 2], 
            label=name, 
            color=colors[name], 
            linewidth=2.5,
            alpha=0.8)
    ax.scatter(path[0, 0], path[0, 1], path[0, 2], 
               color=colors[name], 
               marker='x', s=100, 
               label=f'{name} (Inicio Puro)')

# Marcamos el punto final T1 puro como referencia
ax.scatter(features_muerto[0], features_muerto[1], features_muerto[2], 
           c='cyan', marker='*', s=300, 
           label='Punto Final (Muerte T1 Puro)',
           edgecolor='black')

ax.set_xlabel('H_Z (Desorden Z)', fontsize=12)
ax.set_ylabel('E_X (Correlación X)', fontsize=12)
ax.set_zlabel('H_q (Entrelaz. Prom.)', fontsize=12)
ax.set_title('Mapa de Robustez: Trayectorias con Ruido Mixto (80%T1+20%T2, N=5)', fontsize=16) # <-- Título actualizado
ax.legend()
plt.show()

In [ ]:
3.4